In [69]:
import geopandas as gpd
import pandas as pd
import folium
from matplotlib import pyplot as plt
from pyproj import Geod
from shapely.geometry import LineString
import os

In [107]:
routes = pd.read_csv('/Users/patnaik/Documents/30DayMapChallenge/Day 2 - Lines/sna_nonstop_routes_airlines_2026.csv')
routes.head()

routes = routes[
    ~(
        (routes["destination"] == "AUS") &
        (routes["airline"] == "Allegiant")
    )
].copy()
routes = routes[
    ~routes["destination"].isin(["PIT", "CVG"])
].copy()

In [108]:
aviationdata = pd.read_csv('/Users/patnaik/Documents/30DayMapChallenge/Day 2 - Lines/NTAD_Aviation_Facilities_-1138898284069410957.csv')
aviationdata.head()

/var/folders/m3/1c1bw_hs4sb0s2lvtydzcjkc0000gn/T/ipykernel_53128/2483810402.py:1: DtypeWarning: Columns (0: ALT_FSS_ID, 1: ALT_FSS_NAME, 2: ALT_TOLL_FREE_NO) have mixed types. Specify dtype option on import or set low_memory=False.
  aviationdata = pd.read_csv('/Users/patnaik/Documents/30DayMapChallenge/Day 2 - Lines/NTAD_Aviation_Facilities_-1138898284069410957.csv')


,OBJECTID,EFF_DATE,SITE_NO,SITE_TYPE_CODE,STATE_CODE,ARPT_ID,CITY,COUNTRY_CODE,REGION_CODE,ADO_CODE,...,TRNS_STRG_HGR_FLAG,TRNS_STRG_TIE_FLAG,OTHER_SERVICES,WIND_INDCR_FLAG,ICAO_ID,MIN_OP_NETWORK,USER_FEE_FLAG,CTA,x,y
0,1,2026/10/29,103.00,A,AL,0J0,ABBEVILLE,US,ASO,JAN,...,NaN,Y,NaN,Y-L,NaN,N,NaN,NaN,-9.488711e+06,3.711143e+06
1,2,2026/10/29,106.00,A,AL,2A8,ADDISON,US,ASO,JAN,...,NaN,N,NaN,Y,NaN,N,NaN,NaN,-9.702402e+06,4.057996e+06
2,3,2026/10/29,108.00,A,AL,AL03,AKRON,US,ASO,JAN,...,NaN,NaN,NaN,Y,NaN,N,NaN,NaN,-9.764265e+06,3.875080e+06
3,4,2026/10/29,110.00,A,AL,EET,ALABASTER,US,ASO,JAN,...,NaN,Y,"INSTR,RNTL,SALES",Y-L,KEET,N,NaN,NaN,-9.660664e+06,3.918925e+06
4,5,2026/10/29,110.01,H,AL,AL39,ALABASTER,US,ASO,JAN,...,NaN,NaN,NaN,Y,NaN,N,NaN,NaN,-9.663892e+06,3.928830e+06


In [109]:
airport_codes = pd.concat([routes["destination"],routes["origin"]]).unique()

airports = aviationdata[
    aviationdata["ARPT_ID"].isin(airport_codes)
][["ARPT_ID", "ARPT_NAME", "CITY", "LAT_DECIMAL", "LONG_DECIMAL"]]


In [111]:
# There are 6 missing airports, 2 in Arizona, and 4 international airports not included in the Bureau of Transportation Statistics dataset
arizona = aviationdata[
    aviationdata["ARPT_ID"].isin(["IWA", "SDL"])
][["ARPT_ID", "ARPT_NAME", "CITY", "LAT_DECIMAL", "LONG_DECIMAL"]].copy()

arizona["ARPT_ID"] = arizona["ARPT_ID"].replace({
    "IWA": "AZA",
    "SDL": "SCF"
})

airports = pd.concat([airports, arizona], ignore_index=True)
international = pd.DataFrame({
    "ARPT_ID": ["PVR", "YVR", "SJD", "YYC"],
    "ARPT_NAME": [
        "LICENCIADO GUSTAVO DÍAZ ORDAZ INTL",
        "VANCOUVER INTL",
        "LOS CABOS INTL",
        "CALGARY INTL"
    ],
    "CITY": [
        "PUERTO VALLARTA",
        "VANCOUVER",
        "SAN JOSÉ DEL CABO",
        "CALGARY"
    ],
    "LAT_DECIMAL": [
        20.680010, 49.193956, 23.155245,  51.131337
    ],
    "LONG_DECIMAL": [
        -105.251611, -123.183034, -109.718957, -114.016538
    ]
})
airports = pd.concat(
    [airports, international],
    ignore_index=True
)
airports

,ARPT_ID,ARPT_NAME,CITY,LAT_DECIMAL,LONG_DECIMAL
0,PHX,PHOENIX SKY HARBOR INTL,PHOENIX,33.434278,-112.011583
1,TUS,TUCSON INTL,TUCSON,32.117070,-110.941463
2,MRY,MONTEREY RGNL,MONTEREY,36.586952,-121.842783
3,APC,NAPA COUNTY,NAPA,38.213189,-122.280687
4,OAK,OAKLAND SAN FRANCISCO BAY,OAKLAND,37.721261,-122.221151
5,SMF,SACRAMENTO INTL,SACRAMENTO,38.695444,-121.590778
6,SFO,SAN FRANCISCO INTL,SAN FRANCISCO,37.618806,-122.375417
7,SJC,MINETA SAN JOSE INTL,SAN JOSE,37.362995,-121.928621
8,SNA,JOHN WAYNE/ORANGE COUNTY,SANTA ANA,33.675662,-117.868233
9,STS,CHARLES M SCHULZ/SONOMA COUNTY,SANTA ROSA,38.509694,-122.812889


In [112]:
airports_gdf = gpd.GeoDataFrame(
    airports,
    geometry=gpd.points_from_xy(
        airports["LONG_DECIMAL"],
        airports["LAT_DECIMAL"]
    ),
    crs="EPSG:4326"
)

In [113]:
routes_gdf = routes.merge(
    airports_gdf,
    left_on="destination",
    right_on="ARPT_ID",
    how="left"
)

routes_gdf.head()

,origin,destination,destination_city,airline,seasonal,ARPT_ID,ARPT_NAME,CITY,LAT_DECIMAL,LONG_DECIMAL,geometry
0,SNA,ATW,Appleton,Allegiant,True,ATW,APPLETON INTL,APPLETON,44.258093,-88.519072,POINT (-88.51907 44.25809)
1,SNA,ATL,Atlanta,Delta,False,ATL,HARTSFIELD/JACKSON ATLANTA INTL,ATLANTA,33.636700,-84.427864,POINT (-84.42786 33.6367)
2,SNA,AUS,Austin,Southwest,False,AUS,AUSTIN-BERGSTROM INTL,AUSTIN,30.194527,-97.669876,POINT (-97.66988 30.19453)
3,SNA,BOI,Boise,Alaska,False,BOI,BOISE AIR TRML/GOWEN FLD,BOISE,43.564361,-116.222861,POINT (-116.22286 43.56436)
4,SNA,BOI,Boise,Allegiant,False,BOI,BOISE AIR TRML/GOWEN FLD,BOISE,43.564361,-116.222861,POINT (-116.22286 43.56436)


In [114]:
routes_grouped = (
    routes_gdf.groupby("destination")
    .agg({
        "destination_city": "first",
        "airline": lambda x: ", ".join(sorted(x.unique())),
        "seasonal": "all",
        "geometry": "first"
    })
    .reset_index()
)

routes_grouped = gpd.GeoDataFrame(
    routes_grouped,
    geometry="geometry",
    crs="EPSG:4326"
)

routes_grouped

,destination,destination_city,airline,seasonal,geometry
0,APC,Napa County,JSX,True,POINT (-122.28069 38.21319)
1,ATL,Atlanta,Delta,False,POINT (-84.42786 33.6367)
2,ATW,Appleton,Allegiant,True,POINT (-88.51907 44.25809)
3,AUS,Austin,Southwest,False,POINT (-97.66988 30.19453)
4,AZA,Mesa,Allegiant,False,POINT (-111.65546 33.30782)
5,BOI,Boise,"Alaska, Allegiant",False,POINT (-116.22286 43.56436)
6,CLT,Charlotte,American,False,POINT (-80.95138 35.21319)
7,CMH,Columbus,Breeze,True,POINT (-82.89216 39.99695)
8,COS,Colorado Springs,Allegiant,True,POINT (-104.70078 38.80582)
9,DAL,Dallas Love Field,Southwest,False,POINT (-96.85088 32.84594)


In [115]:
geod = Geod(ellps="WGS84")

sna = airports_gdf.loc[
    airports_gdf["ARPT_ID"] == "SNA", "geometry"
].iloc[0]

def create_flight_path(destination):
    points = geod.npts(
        sna.x, sna.y,
        destination.x, destination.y,
        50
    )

    coordinates = (
        [(sna.x, sna.y)]
        + points
        + [(destination.x, destination.y)]
    )

    return LineString(coordinates)

In [116]:
flight_paths = routes_grouped.copy()

flight_paths["geometry"] = flight_paths.geometry.apply(
    create_flight_path
)

flight_paths = gpd.GeoDataFrame(
    flight_paths,
    geometry="geometry",
    crs="EPSG:4326"
)

flight_paths

,destination,destination_city,airline,seasonal,geometry
0,APC,Napa County,JSX,True,"LINESTRING (-117.86823 33.67566, -117.95008 33..."
1,ATL,Atlanta,Delta,False,"LINESTRING (-117.86823 33.67566, -117.22338 33..."
2,ATW,Appleton,Allegiant,True,"LINESTRING (-117.86823 33.67566, -117.37716 33..."
3,AUS,Austin,Southwest,False,"LINESTRING (-117.86823 33.67566, -117.4593 33...."
4,AZA,Mesa,Allegiant,False,"LINESTRING (-117.86823 33.67566, -117.74598 33..."
5,BOI,Boise,"Alaska, Allegiant",False,"LINESTRING (-117.86823 33.67566, -117.83992 33..."
6,CLT,Charlotte,American,False,"LINESTRING (-117.86823 33.67566, -117.17263 33..."
7,CMH,Columbus,Breeze,True,"LINESTRING (-117.86823 33.67566, -117.24843 33..."
8,COS,Colorado Springs,Allegiant,True,"LINESTRING (-117.86823 33.67566, -117.6265 33...."
9,DAL,Dallas Love Field,Southwest,False,"LINESTRING (-117.86823 33.67566, -117.45495 33..."


In [120]:

m = folium.Map(location=[39,-105],zoom_start=4,tiles=None,width="100%",height=750)

folium.TileLayer(
    tiles="https://basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}.png?key=cb1_4zt2_1_ccb18612c3bd332fbb7e7e5d",
    attr="© CARTO, © OpenStreetMap contributors",
    name="CARTO Voyager",
    overlay=True,
    control=False
).add_to(m)

all_routes = folium.FeatureGroup(name="All Airlines",show=True,overlay=False)

for _,row in flight_paths.iterrows():
    folium.GeoJson(
        row["geometry"].__geo_interface__,
        control=False,
        style_function=lambda feature: {"color":"navy","weight":2,"opacity":0.7},
        tooltip=folium.Tooltip(
            f"SNA → {row['destination']}<br>"
            f"Destination: {row['destination_city']}<br>"
            f"Airlines: {row['airline']}"
        )
    ).add_to(all_routes)

all_routes.add_to(m)

for _,row in routes_grouped.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y,row.geometry.x],
        radius=5,
        color="#007F86",
        weight=2,
        fill=True,
        fill_color="white",
        fill_opacity=1,
        tooltip=folium.Tooltip(
            f"<b>{row['destination_city']} ({row['destination']})</b><br>"
            f"Airlines: {row['airline']}"
        ),
        popup=folium.Popup(
            f"<b>{row['destination_city']} ({row['destination']})</b><br>"
            f"Airlines: {row['airline']}",
            max_width=300
        )
    ).add_to(m)

folium.CircleMarker(
    location=[sna.y,sna.x],
    radius=7,
    color="red",
    fill=True,
    fill_color="red",
    fill_opacity=1,
    tooltip="John Wayne Airport (SNA)"
).add_to(m)

airlines = sorted(routes["airline"].unique())

for airline in airlines:
    layer = folium.FeatureGroup(name=airline,show=False,overlay=False)

    selected = flight_paths[
        flight_paths["destination"].isin(
            routes.loc[routes["airline"]==airline,"destination"]
        )
    ]

    for _,row in selected.iterrows():
        folium.GeoJson(
            row.geometry.__geo_interface__,
            control=False,
            style_function=lambda feature: {"color":"navy","weight":3,"opacity":0.9},
            tooltip=folium.Tooltip(
                f"SNA → {row['destination']}<br>Airline: {airline}"
            )
        ).add_to(layer)

    layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m


In [121]:
m.save("NonstopFromSNA.html")